# 🏦 Financial Inclusion — Baseline Model (Logistic Regression)

This notebook takes the encoded feature table and builds a baseline logistic regression model.

**Three preprocessing steps before modelling:**
1. Separate features (X) from the target (y)
2. Scale `age_of_respondent` — the only column not already on a 0–1 scale
3. Handle class imbalance — only 14% of respondents have a bank account

**Answer to 'how do the many job columns work?'**  
You pass all columns into the model at once as X. Logistic regression assigns a separate weight (coefficient) to every column — including each of the 9 job dummy columns. Together they describe the contribution of each job type. Nothing special needed; sklearn handles it automatically.

---

## 📦 Step 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

print('✅ Imports done')

---
## 📂 Step 1 — Load the Data

In [ ]:
df = pd.read_csv('features_encoded_train.csv')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head(3)

---
## ⚠️ Step 2 — Check Class Imbalance

Before splitting, we check how balanced the target is. This matters because:
- If 86% of rows are class 0, a model that always predicts 0 gets 86% accuracy — but is useless
- We need metrics like **ROC-AUC** that are not fooled by class imbalance

In [ ]:
target_col = 'target_bank_account'

counts = df[target_col].value_counts()
pcts = df[target_col].value_counts(normalize=True) * 100

print('Target distribution:')
print(f'  No bank account (0): {counts[0]:,}  ({pcts[0]:.1f}%)')
print(f'  Has bank account (1): {counts[1]:,}  ({pcts[1]:.1f}%)')
print(f'\n⚠️  Class imbalance ratio: {counts[0]/counts[1]:.1f}:1')
print('   → We will use class_weight="balanced" in the model to compensate.')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['No bank account', 'Has bank account'], counts.values,
       color=['#5F5E5A', '#185FA5'], edgecolor='white')
for i, (v, p) in enumerate(zip(counts.values, pcts.values)):
    ax.text(i, v + 100, f'{v:,} ({p:.1f}%)', ha='center', fontsize=9, color='#5F5E5A')
ax.set_title('Class distribution', fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## ✂️ Step 3 — Separate X and y, then Split

- **X** = all 17 feature columns (everything except the target)
- **y** = `target_bank_account` (what we want to predict)
- We split: 80% for training, 20% for validation
- `stratify=y` ensures both splits have the same 86/14 class ratio

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

print(f'Features (X): {X.shape[1]} columns')
print(f'Target  (y): {y.name}')
print(f'\nFeature columns:')
for col in X.columns:
    print(f'  • {col}')

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y      # preserve the class ratio in both splits
)

print(f'Training set  : {X_train.shape[0]:,} rows')
print(f'Validation set: {X_val.shape[0]:,} rows')
print(f'\nClass balance in train: {y_train.mean()*100:.1f}% positive')
print(f'Class balance in val  : {y_val.mean()*100:.1f}% positive')

---
## 📏 Step 4 — Scale Age

Logistic regression works best when all features are on a similar scale. Age (16–100) is much larger than all other columns (0–4 at most). We use `StandardScaler` which transforms a column to have mean=0 and std=1.

> **Important rule**: fit the scaler on training data only, then apply it to both train and validation. If we fitted on all data, we'd leak information from the validation set.

In [ ]:
# Only age needs scaling — all other columns are already 0/1 or 0–4
cols_to_scale = ['age_of_respondent']

scaler = StandardScaler()

# Fit ONLY on training data
scaler.fit(X_train[cols_to_scale])

# Apply to both train and validation
X_train = X_train.copy()
X_val = X_val.copy()

X_train[cols_to_scale] = scaler.transform(X_train[cols_to_scale])
X_val[cols_to_scale]   = scaler.transform(X_val[cols_to_scale])

print(f'age_of_respondent — after scaling:')
print(f'  Train: mean={X_train["age_of_respondent"].mean():.4f}, std={X_train["age_of_respondent"].std():.4f}')
print(f'  Val  : mean={X_val["age_of_respondent"].mean():.4f}, std={X_val["age_of_respondent"].std():.4f}')
print('\n✅ Scaling done. All other columns unchanged.')

---
## 🤖 Step 5 — Train the Logistic Regression

Key settings:
- `class_weight='balanced'` — automatically upweights the minority class (bank account = yes)
- `max_iter=1000` — more iterations to ensure convergence
- All 17 features are passed as X — the model learns a coefficient for every single column

In [ ]:
model = LogisticRegression(
    class_weight='balanced',  # handles the 86/14 imbalance
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

print('✅ Model trained')
print(f'   Features used: {X_train.shape[1]}')
print(f'   Training rows: {X_train.shape[0]:,}')

---
## 📊 Step 6 — Evaluate on Validation Set

We evaluate using:
- **ROC-AUC** — the primary metric; measures how well the model separates the two classes (1.0 = perfect, 0.5 = random)
- **Classification report** — precision, recall, F1 per class
- **Confusion matrix** — shows how many true/false positives and negatives

In [ ]:
# Predictions
y_pred = model.predict(X_val)              # hard labels (0 or 1)
y_proba = model.predict_proba(X_val)[:, 1] # probability of class 1

# ROC-AUC
auc = roc_auc_score(y_val, y_proba)
print(f'ROC-AUC score: {auc:.4f}')
print()

# Classification report
print('Classification report (validation set):')
print(classification_report(y_val, y_pred, target_names=['No bank account', 'Has bank account']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(y_val, y_proba)
axes[0].plot(fpr, tpr, color='#185FA5', lw=2, label=f'Logistic Regression (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline (AUC = 0.500)')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#185FA5')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)
axes[0].spines[['top','right']].set_visible(False)

# --- Confusion Matrix ---
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['No bank account', 'Has bank account'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix (validation set)', fontweight='bold')

plt.suptitle('Baseline Model Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔍 Step 7 — Feature Coefficients

Logistic regression is fully interpretable: each feature gets a coefficient.
- **Positive coefficient** → increases the probability of having a bank account
- **Negative coefficient** → decreases it
- **Larger absolute value** → stronger effect

This is where the one-hot job columns pay off — you can read off exactly which job type is most associated with bank account ownership.

In [ ]:
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

# Clean up labels for readability
coef_df['label'] = (coef_df['feature']
    .str.replace('ohe_job_', 'job: ', regex=False)
    .str.replace('ohe_country_', 'country: ', regex=False)
    .str.replace('bin_location_type', 'Urban location', regex=False)
    .str.replace('bin_cellphone_access', 'Cellphone access', regex=False)
    .str.replace('bin_gender', 'Gender (Male=1)', regex=False)
    .str.replace('ord_education_level', 'Education level', regex=False)
    .str.replace('age_of_respondent', 'Age', regex=False)
)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#185FA5' if v >= 0 else '#993C1D' for v in coef_df['coefficient']]
bars = ax.barh(coef_df['label'], coef_df['coefficient'],
               color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('Feature coefficients — logistic regression\n(positive = increases bank account probability)',
             fontweight='bold', pad=12)
ax.set_xlabel('Coefficient value')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print('\nTop 5 positive (most associated with having a bank account):')
print(coef_df.head(5)[['label','coefficient']].to_string(index=False))
print('\nTop 5 negative (most associated with NOT having a bank account):')
print(coef_df.tail(5)[['label','coefficient']].to_string(index=False))

---
## 📋 Step 8 — Summary

This is your **baseline**. Any more complex model (Random Forest, XGBoost) should beat this ROC-AUC to justify the added complexity.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print('=' * 50)
print('  BASELINE MODEL SUMMARY')
print('=' * 50)
print(f'  Model          : Logistic Regression')
print(f'  Features used  : {X_train.shape[1]}')
print(f'  Train rows     : {X_train.shape[0]:,}')
print(f'  Val rows       : {X_val.shape[0]:,}')
print(f'  Class weight   : balanced')
print()
print(f'  ROC-AUC        : {auc:.4f}')
print(f'  Accuracy       : {accuracy_score(y_val, y_pred):.4f}')
print(f'  Precision (1)  : {precision_score(y_val, y_pred):.4f}')
print(f'  Recall (1)     : {recall_score(y_val, y_pred):.4f}')
print(f'  F1 (1)         : {f1_score(y_val, y_pred):.4f}')
print('=' * 50)
print()
print('Next steps to try:')
print('  → Random Forest or XGBoost (handles non-linearity)')
print('  → Tune decision threshold (default 0.5 may not be optimal)')
print('  → Cross-validation instead of single train/val split')
print('  → SMOTE oversampling as an alternative to class_weight')